# 03 Classical TF-IDF Models

Generated from `notebooks/ledgar_clause_classification_pipeline.ipynb`.

Source cell indices: `1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 25, 26, 27, 28, 29`.

- Runs TF-IDF Logistic Regression, Linear SVM, and Naive Bayes experiments.


## 1. Colab Setup, Imports, and Configuration

This stage prepares the runtime so the same notebook can run locally or in Google Colab. In Colab, upload, unzip, sync, or clone the whole project folder, not just this notebook.



Importing Libraries and Modules

In [5]:
from pathlib import Path

import importlib.util
import os
import subprocess
import sys
import numpy as np
import pandas as pd

import json
import math
import re

import pandas as pd
from datetime import datetime, timezone
from IPython.display import display

File Setup

In [6]:
PROJECT_ROOT_OVERRIDE = os.environ.get("LEDGAR_PROJECT_ROOT", "").strip()
AUTO_MOUNT_GOOGLE_DRIVE = True
INSTALL_REQUIREMENTS_IN_COLAB = True


def running_in_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or importlib.util.find_spec("google.colab") is not None


IN_COLAB = running_in_colab()

In [7]:

if IN_COLAB:
    print("Google Colab runtime detected.")

if IN_COLAB and AUTO_MOUNT_GOOGLE_DRIVE:
    try:
        if Path("/content/drive/MyDrive").exists():
            print("Google Drive is already available.")
        else:
            from google.colab import drive

            drive.mount("/content/drive")
    except Exception as exc:
        print(f"Google Drive mount skipped/failed: {type(exc).__name__}: {exc}")


def looks_like_project_root(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "modules").is_dir()


def project_root_candidates_near(path: Path) -> list[Path]:
    path = path.expanduser()
    candidates = [path, *path.parents]
    if path.exists() and path.is_dir():
        for pattern in (
            "pyproject.toml",
            "*/pyproject.toml",
            "*/*/pyproject.toml",
            "*/*/*/pyproject.toml",
        ):
            candidates.extend(pyproject.parent for pyproject in path.glob(pattern))
    deduped = []
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            resolved = candidate
        if resolved not in seen:
            deduped.append(resolved)
            seen.add(resolved)
    return deduped


def parent_search(start: Path) -> Path | None:
    for candidate in project_root_candidates_near(start):
        if looks_like_project_root(candidate):
            return candidate
    return None


def common_colab_candidates() -> list[Path]:
    candidates = [
        Path("/content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing"),
    ]
    for base in (Path("/content"), Path("/content/drive/MyDrive")):
        if base.exists():
            for pattern in (
                "Natural-Language-Processing",
                "*/Natural-Language-Processing",
                "*/*/Natural-Language-Processing",
                "*/*/*/Natural-Language-Processing",
            ):
                candidates.extend(base.glob(pattern))
    return candidates


def find_notebook_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE:
        override = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        for candidate in project_root_candidates_near(override):
            if looks_like_project_root(candidate):
                if candidate != override:
                    print(f"PROJECT_ROOT_OVERRIDE pointed to a parent folder; using nested project root: {candidate}")
                return candidate
        raise FileNotFoundError(
            f"PROJECT_ROOT_OVERRIDE does not contain pyproject.toml and modules/, and no nested project root was found under it: {override}\n"
            "Check the Drive folder path, or run this diagnostic: list(Path('/content/drive/MyDrive').glob('**/pyproject.toml'))"
        )

    root = parent_search(Path.cwd())
    if root is not None:
        return root

    if IN_COLAB:
        for candidate in common_colab_candidates():
            if candidate.exists() and looks_like_project_root(candidate):
                return candidate.resolve()

    raise FileNotFoundError(
        "Could not find the project root containing pyproject.toml and modules/.\n"
        "In Colab, upload or clone the whole repository, then set PROJECT_ROOT_OVERRIDE "
        "near the top of this cell to that folder. Current working directory: "
        f"{Path.cwd()}"
    )

Google Colab runtime detected.
Mounted at /content/drive


In [8]:
# Find the project root and add it to sys.path so that imports work, even if the notebook is opened in a subfolder or outside the project.
PROJECT_ROOT = find_notebook_project_root()
os.environ["LEDGAR_PROJECT_ROOT"] = str(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# List of (import_name, pip_name) for packages commonly used in notebooks. pip_name can be None if it's the same as import_name.
REQUIRED_NOTEBOOK_PACKAGES = [
    ("pandas", "pandas"),
    ("datasets", "datasets"),
    ("huggingface_hub", "huggingface_hub"),
    ("sklearn", "scikit-learn"),
    ("joblib", "joblib"),
    ("matplotlib", "matplotlib"),
]

# In Colab, install all requirements from requirements-colab.txt if any are missing, to avoid multiple pip installs.
def ensure_notebook_package(import_name: str, pip_name: str | None = None) -> None:
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

# Check for missing imports before installing requirements in Colab, to avoid unnecessary pip installs and speed up notebook startup.
missing_imports = [name for name, _ in REQUIRED_NOTEBOOK_PACKAGES if importlib.util.find_spec(name) is None]
requirements_path = PROJECT_ROOT / "requirements-colab.txt"


if IN_COLAB and INSTALL_REQUIREMENTS_IN_COLAB and requirements_path.exists() and missing_imports:
    print(f"Installing Colab requirements from {requirements_path}.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)])
else:
    for import_name, pip_name in REQUIRED_NOTEBOOK_PACKAGES:
        ensure_notebook_package(import_name, pip_name)

Custom Modules and Libraries

In [9]:
from modules.data_setup import (
    adapt_cuad_to_clause_classification,
    build_project_paths,
    download_cuad_if_missing,
    load_cuad_raw_files,
    load_or_download_ledgar,
    normalise_whitespace,
    print_dataset_availability,
    seed_everything,
)
from modules.preprocessing import (
    bpe_encode_text,
    bpe_encode_word,
    clean_html_entities,
    corpus_word_frequencies,
    create_ledgar_eda,
    legal_safe_tokenise,
    negation_aware_tokenise,
    preprocess_ledgar,
    preprocessing_technique_rundown,
    regex_tokenise,
    train_bpe_tokeniser,
    write_preprocessing_rundown,
)
from modules.baselines import run_baseline_experiments
from modules.classical_models import run_classical_experiments
from modules.sequence_model import SequenceModelConfig, train_sequence_classifier
from modules.transformer_model import train_transformer_classifier
from modules.transformer_hpt import TransformerHPTConfig, run_two_stage_transformer_hpt
from modules.qwen_prompting import run_qwen_baseline
from modules.agentic_review import run_agentic_review
from modules.evaluation import save_final_comparison
from modules.error_analysis import run_error_analysis
from modules.wandb_reporting import finish_wandb_run, log_wandb_outputs, start_wandb_run



| Setting | Value | Purpose |
|---|---:|---|
| `SEED` | `42` | Makes sampling, baseline randomness, and train/test helper behavior reproducible. |
| `DATASET_NAME` | `LEDGAR` | Keeps the main experiment scoped to LEDGAR clause classification. |
| `TOP_K_LABELS` | `20` | Restricts the task to the 20 most frequent training labels for a manageable coursework experiment. |
| `RUN_CLASSICAL_MODELS` | `True` | Enables TF-IDF model experiments. |
| `RUN_TRANSFORMER` | `True` | Attempts transformer fine-tuning only when the runtime can support it. |
| `RUN_QWEN_BASELINE` | `True` | Attempts Qwen prompting only when GPU/model loading is available. |
| `RUN_AGENTIC_EXTENSION` | `True` | Enables a small review workflow demonstration, not an autonomous agent. |
| `RUN_WANDB` | `True` | Sends metrics and safe artifacts to W&B when credentials are available. |

Model and feature hyperparameters declared here:

| Component | Hyperparameters |
|---|---|
| TF-IDF search | `max_features` in `[10000, 30000]`; `ngram_range` in `[(1, 1), (1, 2)]`; `lowercase=True`; `stop_words=None` |
| Transformer | `distilbert-base-uncased`; `max_length=256` |
| Optional legal transformer | `nlpaueb/legal-bert-base-uncased` can be substituted manually if GPU resources allow |
| Qwen prompting | `Qwen/Qwen2.5-3B-Instruct`; test sample size `200`; one few-shot example per class when available |
| W&B logging | Uses `WANDB_API_KEY` from Colab Secrets or the environment; text-containing prediction/error tables are not uploaded unless `WANDB_LOG_TEXT_TABLES=True` |

Explainability note: keeping all configuration values in one cell makes it clear which choices affect runtime cost, model capacity, and evaluation scope.

In [10]:
SEED = 42

DATASET_NAME = "LEDGAR"

TOP_K_LABELS = 20

MAX_FEATURES_LIST = [10000, 30000]

NGRAM_RANGES = [(1, 1), (1, 2)]

RUN_CLASSICAL_MODELS = True

RUN_TRANSFORMER = True

RUN_TRANSFORMER_HPT = False

HPT_RANDOM_TRIALS = 8

HPT_BAYES_TRIALS = 8

RUN_QWEN_BASELINE = True

RUN_AGENTIC_EXTENSION = True

RUN_NAIVE_BAYES = True

RUN_SEQUENCE_MODEL = False

RUN_WANDB = True

WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "ledgar-clause-classification")

WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "").strip() or None

WANDB_MODE = os.environ.get("WANDB_MODE", "online")

WANDB_LOG_ARTIFACTS = True

WANDB_LOG_TEXT_TABLES = False

WANDB_LOG_MODEL_FILES = False

TRANSFORMER_MODEL_NAME = "distilbert-base-uncased"

OPTIONAL_LEGAL_MODEL_NAME = "nlpaueb/legal-bert-base-uncased"

QWEN_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

MAX_TRANSFORMER_LENGTH = 256

QWEN_EVAL_SAMPLE_SIZE = 200

QWEN_FEW_SHOT_EXAMPLES_PER_CLASS = 1


DOWNLOAD_LEDGAR_IF_MISSING = True

DOWNLOAD_CUAD_IF_MISSING = True

USE_HF_CACHE = True

FORCE_REDOWNLOAD = False

paths = build_project_paths(PROJECT_ROOT)
DEVICE = seed_everything(SEED)


Weights and Biases Setup

In [11]:
print(f"Project root: {paths.project_root}")
print(f"Colab runtime: {IN_COLAB}")
print(f"Raw LEDGAR directory: {paths.ledgar_raw_dir}")
print(f"Results directory: {paths.results_dir}")
print(f"Device: {DEVICE}")

Project root: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing
Colab runtime: True
Raw LEDGAR directory: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/data/raw/lexglue_ledgar
Results directory: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/results
Device: cuda


In [12]:

try:
    import torch

    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("GPU is unavailable. Transformer/Qwen sections will skip or reduce work gracefully.")
except Exception:
    print("PyTorch is unavailable. Transformer/Qwen sections will skip if they require it.")

wandb_run = start_wandb_run(
    enabled=RUN_WANDB,
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    group="ledgar-coursework",
    tags=["ledgar", "legal-clause-classification", "coursework"],
    config={
        "seed": SEED,
        "dataset_name": DATASET_NAME,
        "top_k_labels": TOP_K_LABELS,
        "run_classical_models": RUN_CLASSICAL_MODELS,
        "run_transformer": RUN_TRANSFORMER,
        "run_transformer_hpt": RUN_TRANSFORMER_HPT,
        "hpt_random_trials": HPT_RANDOM_TRIALS,
        "hpt_bayes_trials": HPT_BAYES_TRIALS,
        "run_qwen_baseline": RUN_QWEN_BASELINE,
        "run_agentic_extension": RUN_AGENTIC_EXTENSION,
        "run_naive_bayes": RUN_NAIVE_BAYES,
        "run_sequence_model": RUN_SEQUENCE_MODEL,
        "transformer_model_name": TRANSFORMER_MODEL_NAME,
        "max_transformer_length": MAX_TRANSFORMER_LENGTH,
        "qwen_model_name": QWEN_MODEL_NAME,
        "qwen_eval_sample_size": QWEN_EVAL_SAMPLE_SIZE,
        "device": str(DEVICE),
        "log_text_tables": WANDB_LOG_TEXT_TABLES,
        "log_model_files": WANDB_LOG_MODEL_FILES,
    },
    mode=WANDB_MODE,
)

WANDB_ACTIVE = wandb_run is not None


CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
W&B Colab Secrets lookup skipped: TimeoutException: Requesting secret WANDB_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.
W&B logging skipped: no WANDB_API_KEY or ~/.netrc credentials found.
Add a Colab Secret named WANDB_API_KEY, run wandb.login(), or set WANDB_API_KEY, then rerun the notebook.


## 2. Dataset Download and Raw Setup

This stage obtains the raw datasets without training or preprocessing models. LEDGAR remains the main classification dataset. CUAD is treated separately because it is structured as a contract-review question-answering/span-extraction dataset rather than a direct clause-classification dataset.

Data governance choices:

- LEDGAR is loaded from Hugging Face with `load_dataset("coastalcph/lex_glue", "ledgar")` when local JSONL files are missing.
- Official LEDGAR train, validation, and test splits are preserved when available.
- Raw LEDGAR split exports are saved under `data/raw/lexglue_ledgar/` as JSONL files.
- CUAD raw files are downloaded from `theatticusproject/cuad` when available, but CUAD is not merged with LEDGAR.
- If CUAD is missing, the notebook prints a clear message and continues with LEDGAR.

Inputs and outputs:

| Input | Output |
|---|---|
| Hugging Face LEDGAR or local JSONL | `ledgar_raw_splits` dictionary with train/validation/test DataFrames |
| Optional CUAD raw files | `cuad_clause_df` containing extracted span-level examples for optional inspection |

This stage deliberately does not select labels, encode classes, train models, or compute metrics.


CAUD Dataset

In [13]:
ledgar_raw_splits = load_or_download_ledgar(
    paths,
    download_if_missing=DOWNLOAD_LEDGAR_IF_MISSING,
    force_redownload=FORCE_REDOWNLOAD,
)

cuad_json_path, master_clauses_path = download_cuad_if_missing(
    paths,
    download_if_missing=DOWNLOAD_CUAD_IF_MISSING,
    force_redownload=FORCE_REDOWNLOAD,
)

raw_cuad_json, master_clauses_df = load_cuad_raw_files(cuad_json_path, master_clauses_path)
cuad_clause_df = adapt_cuad_to_clause_classification(raw_cuad_json)
print_dataset_availability(ledgar_raw_splits, cuad_json_path, master_clauses_path, cuad_clause_df)

Loading LEDGAR from data/raw/lexglue_ledgar JSONL files.
LEDGAR train: 60000 rows, columns=['text', 'label']
LEDGAR validation: 10000 rows, columns=['text', 'label']
LEDGAR test: 10000 rows, columns=['text', 'label']
Using existing CUAD files from data/raw/cuad/.

Dataset availability:
- LEDGAR downloaded/loaded: yes
- LEDGAR train size: 60000
- LEDGAR validation size: 10000
- LEDGAR test size: 10000
- CUAD JSON found: yes
- CUAD master clauses CSV found: yes
- CUAD adapted span examples available for optional analysis: 13062


## 3. LEDGAR Preprocessing and EDA

This stage is intentionally split into small cells so each lecture/lab technique can be inspected before the full dataset is processed.

Preprocessing changes the raw text into a cleaner form. Feature extraction converts text into numeric vectors. Modelling starts only after those vectors are produced.


### Cell 1 - Separate preprocessing, feature extraction, and modelling

| Stage | What happens here | Examples in this notebook |
|---|---|---|
| Preprocessing | Clean or tokenise raw clause text | HTML/entity cleanup, whitespace normalisation, regex tokens, negation-aware tokens, BPE inspection |
| Feature extraction | Convert text/tokens into numeric vectors | BoW, TF-IDF, unigrams, bigrams |
| Modelling | Fit a classifier using features and labels | Logistic Regression, Linear SVM, Naive Bayes |

Logistic Regression is therefore not preprocessing; it appears later as a model.


In [14]:
from modules.preprocessing import (
    bpe_encode_text,
    bpe_encode_word,
    clean_html_entities,
    corpus_word_frequencies,
    create_ledgar_eda,
    legal_safe_tokenise,
    negation_aware_tokenise,
    preprocess_ledgar,
    preprocessing_technique_rundown,
    regex_tokenise,
    train_bpe_tokeniser,
    write_preprocessing_rundown,
)


In [15]:
# Cell 3 - Show one raw contract clause example before preprocessing.
if ledgar_raw_splits:
    raw_train_df = ledgar_raw_splits["train"]
    raw_text_column = next(column for column in ("text", "provision", "clause", "contract_text") if column in raw_train_df.columns)
    raw_clause = str(raw_train_df.iloc[0][raw_text_column])
else:
    raw_text_column = "text"
    raw_clause = "The Borrower shall not be liable for any indirect damages &amp; shall give notice under Section 5.1."

print(raw_clause[:1000])


Except as otherwise set forth in this Debenture, the Company, for itself and its legal representatives, successors and assigns, expressly waives presentment, protest, demand, notice of dishonor, notice of nonpayment, notice of maturity, notice of protest, presentment for the purpose of accelerating maturity, and diligence in collection.


In [16]:
# Cell 4 - Apply Week 4 lab-style HTML/entity cleanup.
html_clean_clause = clean_html_entities(raw_clause)
print(html_clean_clause[:1000])


Except as otherwise set forth in this Debenture, the Company, for itself and its legal representatives, successors and assigns, expressly waives presentment, protest, demand, notice of dishonor, notice of nonpayment, notice of maturity, notice of protest, presentment for the purpose of accelerating maturity, and diligence in collection.


In [17]:
# Cell 5 - Apply whitespace normalisation.
whitespace_clean_clause = normalise_whitespace(html_clean_clause)
print(whitespace_clean_clause[:1000])


Except as otherwise set forth in this Debenture, the Company, for itself and its legal representatives, successors and assigns, expressly waives presentment, protest, demand, notice of dishonor, notice of nonpayment, notice of maturity, notice of protest, presentment for the purpose of accelerating maturity, and diligence in collection.


In [18]:
# Cell 6 - Run Week 2 regex tokenisation.
regex_tokens = regex_tokenise(whitespace_clean_clause)
print(regex_tokens[:80])
print(f"Token count: {len(regex_tokens)}")


['Except', 'as', 'otherwise', 'set', 'forth', 'in', 'this', 'Debenture', 'the', 'Company', 'for', 'itself', 'and', 'its', 'legal', 'representatives', 'successors', 'and', 'assigns', 'expressly', 'waives', 'presentment', 'protest', 'demand', 'notice', 'of', 'dishonor', 'notice', 'of', 'nonpayment', 'notice', 'of', 'maturity', 'notice', 'of', 'protest', 'presentment', 'for', 'the', 'purpose', 'of', 'accelerating', 'maturity', 'and', 'diligence', 'in', 'collection']
Token count: 47


In [19]:
# Cell 7 - Run legal-safe lowercased tokenisation.
# This lowercases tokens for feature extraction without overwriting the stored clause text.
legal_tokens = legal_safe_tokenise(whitespace_clean_clause)
print(legal_tokens[:80])


['except', 'as', 'otherwise', 'set', 'forth', 'in', 'this', 'debenture', 'the', 'company', 'for', 'itself', 'and', 'its', 'legal', 'representatives', 'successors', 'and', 'assigns', 'expressly', 'waives', 'presentment', 'protest', 'demand', 'notice', 'of', 'dishonor', 'notice', 'of', 'nonpayment', 'notice', 'of', 'maturity', 'notice', 'of', 'protest', 'presentment', 'for', 'the', 'purpose', 'of', 'accelerating', 'maturity', 'and', 'diligence', 'in', 'collection']


In [20]:
# Cell 8 - Run Week 2 negation-aware tokenisation.
negation_tokens = negation_aware_tokenise(whitespace_clean_clause)
print(negation_tokens[:100])


['except', 'as', 'otherwise', 'set', 'forth', 'in', 'this', 'debenture', 'the', 'company', 'for', 'itself', 'and', 'its', 'legal', 'representatives', 'successors', 'and', 'assigns', 'expressly', 'waives', 'presentment', 'protest', 'demand', 'notice', 'of', 'dishonor', 'notice', 'of', 'nonpayment', 'notice', 'of', 'maturity', 'notice', 'of', 'protest', 'presentment', 'for', 'the', 'purpose', 'of', 'accelerating', 'maturity', 'and', 'diligence', 'in', 'collection']


In [21]:
# Cell 9 - Show why stopword removal is skipped for legal clauses.
# These words are often treated as stopwords in generic NLP, but they can change legal meaning.
legal_stopword_examples = {"no", "not", "shall", "may", "unless", "except", "without"}
kept_legal_tokens = [token for token in legal_tokens if token in legal_stopword_examples]

print("Legal stopword-like tokens kept:", kept_legal_tokens)
print("Default decision: do not remove stopwords for contract clause classification.")


Legal stopword-like tokens kept: ['except']
Default decision: do not remove stopwords for contract clause classification.


In [22]:
# Cell 10 - Train a small Week 2/3 BPE tokenizer on training clause samples.
if ledgar_raw_splits:
    bpe_training_texts = ledgar_raw_splits["train"][raw_text_column].astype(str).head(250).tolist()
else:
    bpe_training_texts = [whitespace_clean_clause]

bpe_word_counts = corpus_word_frequencies(bpe_training_texts, max_words=2000)
bpe_merges, bpe_vocab = train_bpe_tokeniser(bpe_word_counts, num_merges=50)

print(f"BPE training words: {len(bpe_word_counts)}")
print(f"BPE merges learned: {len(bpe_merges)}")
print(list(bpe_merges.items())[:10])


BPE training words: 2000
BPE merges learned: 50
[(('e', '</w>'), 0), (('t', 'h'), 1), (('s', '</w>'), 2), (('a', 'n'), 3), (('e', 'r'), 4), (('d', '</w>'), 5), (('i', 'n'), 6), (('t', '</w>'), 7), (('y', '</w>'), 8), (('o', 'r'), 9)]


In [23]:
# Cell 11 - Run BPE encoding/OOV examples.
for word in ["lowest", "lover", "newly", "unwanted", "indemnification", "xyz"]:
    print(f"{word:20} -> {bpe_encode_word(word, bpe_merges)}")

print("Clause BPE preview:")
print(bpe_encode_text(whitespace_clean_clause, bpe_merges)[:100])


lowest               -> ['l', 'o', 'w', 'e', 's', 't</w>']
lover                -> ['l', 'o', 'v', 'er</w>']
newly                -> ['n', 'e', 'w', 'l', 'y</w>']
unwanted             -> ['u', 'n', 'w', 'an', 't', 'ed</w>']
indemnification      -> ['in', 'd', 'em', 'n', 'i', 'f', 'i', 'c', 'a', 'tion</w>']
xyz                  -> ['x', 'y', 'z']
Clause BPE preview:
['ex', 'c', 'e', 'p', 't</w>', 'a', 's</w>', 'o', 'th', 'er', 'w', 'is', 'e</w>', 's', 'e', 't</w>', 'f', 'or', 'th', 'in</w>', 'th', 'is</w>', 'd', 'e', 'b', 'en', 't', 'u', 'r', 'e</w>', 'the</w>', 'co', 'm', 'p', 'any</w>', 'f', 'or</w>', 'i', 't', 's', 'e', 'l', 'f</w>', 'and</w>', 'i', 'ts</w>', 'l', 'e', 'g', 'al', 're', 'p', 're', 's', 'en', 't', 'a', 'ti', 'v', 'es</w>', 'su', 'c', 'c', 'e', 's', 's', 'or', 's</w>', 'and</w>', 'a', 's', 's', 'i', 'g', 'n', 's</w>', 'ex', 'p', 're', 's', 's', 'l', 'y</w>', 'w', 'a', 'i', 'v', 'es</w>', 'p', 're', 's', 'en', 't', 'm', 'ent</w>', 'p', 'ro', 't', 'e', 's']


In [24]:
# Cell 12 - Preprocess full LEDGAR splits and save processed outputs.
processed_splits, label2id, id2label = preprocess_ledgar(
    ledgar_raw_splits,
    paths,
    top_k_labels=TOP_K_LABELS,
    dataset_name=DATASET_NAME,
)

split_summary = create_ledgar_eda(processed_splits, paths.results_dir)

if processed_splits:
    train_df = processed_splits["train"]
    validation_df = processed_splits["validation"]
    test_df = processed_splits["test"]
    label_names = [id2label[i] for i in sorted(id2label)]
    display(split_summary)
    display(pd.DataFrame({"label": label_names}))
else:
    train_df = validation_df = test_df = pd.DataFrame(
        columns=["text", "label", "label_id", "split", "source_dataset"]
    )
    label_names = []
    print("Main LEDGAR experiment cannot run without LEDGAR data.")


,split,rows,classes
0,train,28587,20
1,validation,4670,20
2,test,4732,20


,label
0,Governing Laws
1,Notices
2,Counterparts
3,Entire Agreements
4,Severability
5,Amendments
6,Survival
7,Assignments
8,Expenses
9,Terms


In [25]:
# Cell 13 - Write a readable rundown of preprocessing and feature techniques.
rundown_path = write_preprocessing_rundown(paths.project_root / "outputs" / "preprocessing_techniques.md")
print(f"Wrote: {rundown_path}")
print(preprocessing_technique_rundown())


Wrote: /content/drive/MyDrive/Colab Notebooks/Education/NLP/Natural-Language-Processing/outputs/preprocessing_techniques.md
# Preprocessing and Feature Extraction Rundown

## Preprocessing used
- HTML/entity cleanup: Week 4 lab cleanup pattern, used before tokenisation.
- Whitespace normalisation: keeps clauses readable while removing layout noise.
- Regex tokenisation: Week 2 lab `\b\w+\b` word-boundary tokenisation.
- Legal-safe lowercasing: used inside tokenisers/vectorisers, without overwriting the stored clause text.
- Negation-aware tokenisation: Week 2 lab idea, using `NOT_` on the token after `no`, `not`, or `never`.
- BPE training/encoding: Weeks 2-3 lab implementation for subword/OOV inspection.

## Feature extraction used
- Bag-of-words inspection: Week 2/3 lab idea for understanding sparse features.
- TF-IDF: Week 2 lecture and Week 3 lab term weighting.
- Unigrams and bigrams: Week 2/3 lecture/lab n-gram feature extraction.

## Deliberately not default preprocessing
- Stop

## 4. Shared Result State

This short stage creates shared containers used by the later model sections.

- `completed_results` stores one row per completed or skipped model run.
- `prediction_tables` stores per-example predictions for error analysis.
- `trained_models` stores reusable fitted model objects when available.

Governance purpose: every model section appends to the same result structure, so the final comparison table is generated from actual run outputs rather than manually entered values.


Variable Initialization

In [26]:
completed_results = []

prediction_tables = {}

trained_models = {}

## 6. Classical TF-IDF Models

This stage trains sparse-text supervised models using TF-IDF features. These models are fast, interpretable at the feature level, and provide strong non-neural baselines for legal text classification.


Selection protocol:

1. Train each configuration on the LEDGAR training split.
2. Select the best configuration using validation macro-F1.
3. Evaluate selected models on the test split once.
4. Save the best classical pipeline and vectorizer artifacts separately.

Explainability note: TF-IDF models are useful for coursework governance because their decisions are linked to sparse lexical features rather than hidden contextual embeddings.


Variable Initialization

In [27]:
# Initialize variables to track the best classical model and its name, which will be updated after running classical experiments.

best_classical_model = None

best_classical_name = None

Training for Classical Machine-Learning Models

Feature extraction hyperparameters:

| Hyperparameter | Values |
|---|---|
| `tokenizer` | `negation_aware` lab-grounded tokenizer |
| `max_features` | `10000`, `30000` |
| `ngram_range` | unigram `(1, 1)`, unigram+bigram `(1, 2)` |
| `lowercase` | handled inside the tokenizer |
| `stop_words` | `None` because legal stopword-like terms can change clause meaning |

The classifier is trained only after these features are built.


### Cell 1 - Feature extraction is not modelling

The cells below inspect Bag-of-Words and TF-IDF features before any classifier is trained. This keeps the Week 3 lab feature work separate from Logistic Regression, Linear SVM, and Naive Bayes.


In [28]:
# Cell 2 - Build Bag-of-Words features with CountVectorizer.
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

feature_sample_texts = train_df["text"].astype(str).head(200).tolist() if not train_df.empty else [
    "Borrower shall not be liable for indirect damages.",
    "Either party may terminate this Agreement on notice.",
]

bow_vectorizer = CountVectorizer(
    tokenizer=legal_safe_tokenise,
    token_pattern=None,
    lowercase=False,
)
bow_features = bow_vectorizer.fit_transform(feature_sample_texts)

print(f"BoW matrix shape: {bow_features.shape}")
print(f"BoW vocabulary size: {len(bow_vectorizer.vocabulary_)}")


BoW matrix shape: (200, 1929)
BoW vocabulary size: 1929


In [29]:
# Cell 3 - Inspect non-zero BoW features for one clause.
feature_names = bow_vectorizer.get_feature_names_out()
first_bow = bow_features[0].tocoo()
first_bow_df = pd.DataFrame({
    "feature": feature_names[first_bow.col],
    "count": first_bow.data,
}).sort_values(["count", "feature"], ascending=[False, True])

print(feature_sample_texts[0])
display(first_bow_df.head(30))


Except as otherwise set forth in this Debenture, the Company, for itself and its legal representatives, successors and assigns, expressly waives presentment, protest, demand, notice of dishonor, notice of nonpayment, notice of maturity, notice of protest, presentment for the purpose of accelerating maturity, and diligence in collection.


,feature,count
24,of,5
23,notice,4
12,and,3
10,for,2
5,in,2
27,maturity,2
20,presentment,2
21,protest,2
8,the,2
29,accelerating,1


In [30]:
# Cell 4 - Build TF-IDF unigram features.
tfidf_unigram_vectorizer = TfidfVectorizer(
    tokenizer=negation_aware_tokenise,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 1),
    max_features=MAX_FEATURES_LIST[0],
)
tfidf_unigram_features = tfidf_unigram_vectorizer.fit_transform(feature_sample_texts)

print(f"TF-IDF unigram matrix shape: {tfidf_unigram_features.shape}")
print(tfidf_unigram_vectorizer.get_feature_names_out()[:40])


TF-IDF unigram matrix shape: (200, 2007)
['00' '000' '01473' '01965' '02' '03' '04' '1' '10' '100' '1000' '10177'
 '105' '11' '12' '13' '14' '1401' '1402' '15' '16' '162' '17' '18' '1801'
 '18100' '1996' '2' '2000' '2013' '2015' '2016' '2017' '2020' '21' '22'
 '23226' '24' '250' '28']


In [31]:
# Cell 5 - Build TF-IDF unigram+bigram features.
tfidf_unibigram_vectorizer = TfidfVectorizer(
    tokenizer=negation_aware_tokenise,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    max_features=MAX_FEATURES_LIST[0],
)
tfidf_unibigram_features = tfidf_unibigram_vectorizer.fit_transform(feature_sample_texts)

print(f"TF-IDF unigram+bigram matrix shape: {tfidf_unibigram_features.shape}")
print(tfidf_unibigram_vectorizer.get_feature_names_out()[:40])


TF-IDF unigram+bigram matrix shape: (200, 10000)
['00' '00 p' '000' '000 on' '000 without' '01473' '01473 and' '01965'
 '01965 attention' '02' '02 d' '02 e' '02 of' '03' '03 and' '04'
 '04 shall' '1' '1 10' '1 11' '1 12' '1 18' '1 1996' '1 2017' '1 3' '1 8'
 '1 above' '1 bella' '1 business' '1 comply' '1 confidentiality'
 '1 either' '1 shall' '10' '10 04' '10 1' '10 2' '10 shall' '100' '11']


In [32]:
# Cell 6 - Compare feature matrix shapes before modelling.
feature_shape_summary = pd.DataFrame([
    {"representation": "BoW unigrams", "rows": bow_features.shape[0], "features": bow_features.shape[1]},
    {"representation": "TF-IDF unigrams", "rows": tfidf_unigram_features.shape[0], "features": tfidf_unigram_features.shape[1]},
    {"representation": "TF-IDF unigrams+bigrams", "rows": tfidf_unibigram_features.shape[0], "features": tfidf_unibigram_features.shape[1]},
])
display(feature_shape_summary)


,representation,rows,features
0,BoW unigrams,200,1929
1,TF-IDF unigrams,200,2007
2,TF-IDF unigrams+bigrams,200,10000


### Cell 7 - Modelling starts below

The next cells train Logistic Regression, Linear SVM, and Naive Bayes using the TF-IDF feature setup. These classifiers are modelling steps, not preprocessing.


In [1]:

# Training with Manual Hyperparameter Tuning

if not RUN_CLASSICAL_MODELS:
    print("RUN_CLASSICAL_MODELS doesn't exist or is set to False.")

else:

    classical_output = run_classical_experiments(

        train_df,
        # Training data for classical models

        validation_df,
        # Validation data for classical models (used for hyperparameter tuning)

        test_df,
        # Test data for classical models

        id2label,
        # Mapping from label IDs to label names

        paths.results_dir,
        # Directory to save results and artifacts

        max_features_list=MAX_FEATURES_LIST,
        # List of max_features values to try for CountVectorizer/TfidfVectorizer

        ngram_ranges=NGRAM_RANGES,
        # List of ngram_range tuples to try for CountVectorizer/TfidfVectorizer

        dataset_name=DATASET_NAME,
        # Name of the dataset (used for logging and artifact naming)

        seed=SEED,
        # Random seed for reproducibility

        run_naive_bayes=RUN_NAIVE_BAYES,
        # Whether to run Naive Bayes models (MultinomialNB, ComplementNB)

        tokenizer_name="negation_aware",
        # Lab-grounded tokenizer used inside TF-IDF feature extraction
    )


# Aggregration of classical model results and prediction tables into the overall results and prediction tables.

    completed_results.extend(classical_output["results"])
    prediction_tables.update(classical_output["prediction_tables"])

# Best Model Selection and Display

    best_classical_model = classical_output["best_model"]
    best_classical_name = classical_output["best_model_name"]



    if best_classical_model is not None:

        trained_models["best_classical"] = best_classical_model

# Print classical machine-Learning Models for
# "Model Name",
# "Accuracy",
# "Macro F1",
# "Weighted F1",
# "Notes" columns

# Only print if there are results to show, otherwise skip to avoid empty table display.

    if classical_output["results"]:

        display(pd.DataFrame(classical_output["results"])[["model_name", "accuracy", "macro_f1", "weighted_f1", "notes"]])

NameError: name 'RUN_CLASSICAL_MODELS' is not defined